In [ ]:
import torch
from pathlib import Path
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-4B"

# Pick a writable working dir depending on where we're running
ON_KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content")
local_model = WORK / MODEL_ID.split("/")[-1]

# Download once per session (skips files that already exist)
snapshot_download(MODEL_ID, local_dir=local_model)

# T4s don't support bf16 well, so use fp16 unless the GPU is Ampere or newer
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
dtype = torch.bfloat16 if use_bf16 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(local_model)
model = AutoModelForCausalLM.from_pretrained(
    local_model,
    torch_dtype=dtype,
    device_map="auto",
)

print("Running on:", "Kaggle" if ON_KAGGLE else "Colab/other")
print("Device map:", model.hf_device_map)

In [1]:
!nvidia-smi

Mon Oct 26 13:37:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----